# 02 — Canonical time-series EDA

Explores canonical telemetry before any feature engineering or modelling.
It runs unchanged for Telecom and Petrobras 3W and reads `SPEC-CORE` only.

It does **not** read labels, fill missing values, remove outliers, produce
anomaly scores, or assume any interval is normal. Everything it emits is
threshold-free evidence for Notebook 03.

## 1. Setup

In [ ]:
import hashlib
import json
import os
import shutil
import sys
import tempfile
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_data_root = (Path("/content/drive/MyDrive/anomaly_detection")
                     if IN_COLAB else Path.home() / "anomaly_detection_data")
DATA_ROOT = Path(os.getenv("ANOMALY_DATA_ROOT")
                 or os.getenv("ANOMALY_DRIVE_ROOT")
                 or default_data_root).expanduser()
default_code_root = (DATA_ROOT / "research" / "milestone1" if IN_COLAB
                     else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
                     else Path.cwd() / "notebooks" / "drive_research")
NOTEBOOK_HOME = Path(os.getenv("ANOMALY_NOTEBOOK_HOME", default_code_root)).expanduser()
if not (NOTEBOOK_HOME / "milestone1_core.py").is_file():
    raise FileNotFoundError(f"milestone1_core.py was not found in {NOTEBOOK_HOME}")
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    CORE_VERSION, GAP_TOLERANCE_FACTOR, new_output_directory, read_json, write_json,
)

EDA_VERSION = "0.6.0"
SECTOR = os.getenv("EDA_SECTOR", os.getenv("ADAPTER_SECTOR", "telecom"))
CANONICAL_RUN_IDS = {"telecom": "telecom_core_v0_10_1_run1",
                     "petrobras_3w": "petrobras_3w_core_v0_10_1_run1"}
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{SECTOR}_eda_v0_6_0_run1")

CORE_ROOT = (DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}"
             / SECTOR / CANONICAL_RUN_ID / "SPEC-CORE")
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / EDA_RUN_ID
figure_cache = tempfile.TemporaryDirectory()
FIGURES = Path(figure_cache.name)

SAMPLE_ENTITIES = int(os.getenv("EDA_SAMPLE_ENTITIES", "6"))
MAX_PLOT_METRICS = int(os.getenv("EDA_MAX_PLOT_METRICS", "8"))
MAX_STATE_PLOTS = int(os.getenv("EDA_MAX_STATE_PLOTS", "2"))
MAX_PLOT_POINTS = int(os.getenv("EDA_MAX_PLOT_POINTS", "3000"))
MAX_TEST_POINTS = int(os.getenv("EDA_MAX_TEST_POINTS", "10000"))
MAX_PANEL_ROWS = int(os.getenv("EDA_MAX_PANEL_ROWS", "750000"))

plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 9})
display(pd.Series({
    "runtime": "Colab + Drive" if IN_COLAB else "local Python",
    "data_root": str(DATA_ROOT),
    "code_root": str(NOTEBOOK_HOME),
    "sector": SECTOR,
    "core_root": str(CORE_ROOT),
    "eda_root": str(EDA_ROOT),
    "maximum_pandas_rows": MAX_PANEL_ROWS,
}, name="value").to_frame())

## 2. Read the canonical contract

The catalogue explains metric names, measurement kinds, units, sampling
modes and expected cadence. A single `valid` view — dropping `invalid`,
null and non-finite values — is defined once so every later query means
exactly the same thing by "an observed value".

In [ ]:
if not CORE_ROOT.is_dir():
    raise FileNotFoundError(f"Run Notebook 01B first: {CORE_ROOT}")

core_manifest = read_json(CORE_ROOT / "manifest.json")
if core_manifest["contract_version"] != CORE_VERSION:
    raise ValueError("Unsupported SPEC-CORE version")

catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
registry = pd.read_parquet(CORE_ROOT / "entity_registry.parquet")
episodes = pd.read_parquet(CORE_ROOT / "observation_episodes.parquet")
gaps = pd.read_parquet(CORE_ROOT / "collection_gaps.parquet")

KIND = dict(zip(catalogue["metric_id"], catalogue["measurement_kind"]))
CADENCE = dict(zip(catalogue["metric_id"], catalogue["expected_cadence_seconds"]))
CONTINUOUS = {"gauge", "bounded_fraction"}

duckdb_spill = tempfile.TemporaryDirectory(prefix="eda-duckdb-")
connection = duckdb.connect()
connection.execute("SET memory_limit = ?", [os.getenv("ANOMALY_DUCKDB_MEMORY_LIMIT", "3GB")])
connection.execute("SET threads = ?", [int(os.getenv("ANOMALY_DUCKDB_THREADS", "2"))])
connection.execute("SET temp_directory = ?", [duckdb_spill.name])
telemetry_glob = str(CORE_ROOT / "telemetry" / "*.parquet").replace("'", "''")
connection.execute(f"CREATE VIEW telemetry AS SELECT * FROM read_parquet('{telemetry_glob}')")
connection.execute("""
    CREATE VIEW valid AS SELECT * FROM telemetry
    WHERE quality_code <> 'invalid' AND value IS NOT NULL AND isfinite(value)
""")


def sql(query):
    return connection.execute(query).df()


display(catalogue)
display(pd.Series(core_manifest["row_counts"], name="rows").to_frame())

## 3. Population structure and quality

An `invalid` row is an attempted observation with no usable value. An absent
row is not an observation. `availability` is the share of episodes in which a
metric was observed at all. Low availability means only that the source did
not provide that metric in many episodes; telemetry alone cannot distinguish
"not installed" from "installed but never reported".

Robust 1st/99th percentiles and the extreme-to-central span expose suspicious
tails without inventing sector limits.

In [ ]:
structure = pd.DataFrame([{
    "rows": core_manifest["row_counts"]["telemetry"],
    "entities": registry["entity_id"].nunique(),
    "episodes": episodes["episode_id"].nunique(),
    "metrics": catalogue["metric_id"].nunique(),
    "first_ts": pd.to_datetime(registry["observed_from"], utc=True).min(),
    "last_ts": pd.to_datetime(registry["observed_to"], utc=True).max(),
    "duplicate_keys": 0,  # enforced by Notebook 01B
}])
display(structure)
assert int(structure.loc[0, "duplicate_keys"]) == 0

series_summary = sql("""
    SELECT entity_id, episode_id, metric_id, count(*) AS rows,
           count(*) FILTER (WHERE quality_code <> 'invalid' AND value IS NOT NULL
                            AND isfinite(value)) AS valid_values,
           avg(CASE WHEN quality_code <> 'invalid' AND value IS NOT NULL
                    AND isfinite(value) THEN 1.0 ELSE 0.0 END) AS valid_rate,
           avg(CASE WHEN quality_code = 'clipped' THEN 1.0 ELSE 0.0 END) AS clipped_rate,
           min(value) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS minimum,
           max(value) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS maximum,
           approx_quantile(value, 0.25) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS q25,
           approx_quantile(value, 0.50) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS series_median,
           approx_quantile(value, 0.75) FILTER (WHERE quality_code <> 'invalid' AND isfinite(value)) AS q75
    FROM telemetry GROUP BY entity_id, episode_id, metric_id
""").merge(catalogue, on="metric_id", how="left").merge(
    episodes[["episode_id", "observed_from", "observed_to"]], on="episode_id", how="left")

span = (pd.to_datetime(series_summary["observed_to"], utc=True)
        - pd.to_datetime(series_summary["observed_from"], utc=True)).dt.total_seconds()
series_summary["expected_rows"] = np.floor(span / series_summary["expected_cadence_seconds"]) + 1
series_summary["coverage"] = series_summary["rows"] / series_summary["expected_rows"]
series_summary["series_iqr"] = series_summary["q75"] - series_summary["q25"]
series_summary["constant_series"] = (
    series_summary["valid_values"].gt(1)
    & series_summary["minimum"].eq(series_summary["maximum"]))

value_summary = sql("""
    SELECT metric_id, count(*) AS observed_values,
           approx_count_distinct(value) AS distinct_values,
           min(value) AS minimum, max(value) AS maximum,
           approx_quantile(value, 0.01) AS q01, approx_quantile(value, 0.50) AS median,
           approx_quantile(value, 0.99) AS q99,
           avg(CASE WHEN value = 0 THEN 1.0 ELSE 0.0 END) AS zero_rate,
           avg(value) AS mean, var_samp(value) AS variance
    FROM valid GROUP BY metric_id
""")

per_metric = series_summary.groupby("metric_id", as_index=False).agg(
    episodes_observed=("episode_id", "nunique"),
    valid_rate_median=("valid_rate", "median"),
    clipped_rate_p90=("clipped_rate", lambda s: s.quantile(0.90)),
    coverage_median=("coverage", "median"),
    constant_series_rate=("constant_series", "mean"),
    rows_per_series_median=("rows", "median"),
    series_median_p10=("series_median", lambda s: s.quantile(0.10)),
    series_median_p90=("series_median", lambda s: s.quantile(0.90)),
    series_iqr_median=("series_iqr", "median"),
)
per_metric["availability"] = per_metric["episodes_observed"] / len(episodes)

metric_evidence = catalogue.merge(value_summary, on="metric_id", how="left").merge(
    per_metric, on="metric_id", how="left")
central = metric_evidence["q99"] - metric_evidence["q01"]
extreme = np.maximum((metric_evidence["maximum"] - metric_evidence["median"]).abs(),
                     (metric_evidence["minimum"] - metric_evidence["median"]).abs())
metric_evidence["extreme_to_central_span"] = np.where(central.gt(0), extreme / central, np.nan)
metric_evidence["variance_to_mean"] = np.where(
    metric_evidence["measurement_kind"].eq("interval_count") & metric_evidence["mean"].gt(0),
    metric_evidence["variance"] / metric_evidence["mean"], np.nan)

display(metric_evidence[[
    "metric_id", "measurement_kind", "unit", "availability", "valid_rate_median",
    "coverage_median", "constant_series_rate", "q01", "median", "q99",
    "extreme_to_central_span", "variance_to_mean",
]].round(4))

print("Metrics with the most extreme finite tails (review; no values changed):")
display(metric_evidence.sort_values("extreme_to_central_span", ascending=False)[[
    "metric_id", "minimum", "q01", "median", "q99", "maximum",
    "extreme_to_central_span",
]].head(12).round(4))

gap_summary = (gaps.assign(
    gap_seconds=(pd.to_datetime(gaps["gap_end"], utc=True)
                 - pd.to_datetime(gaps["gap_start"], utc=True)).dt.total_seconds())
    .groupby("metric_id", as_index=False)
    .agg(gaps=("gap_start", "size"), affected_series=("episode_id", "nunique"),
         median_gap_seconds=("gap_seconds", "median"), max_gap_seconds=("gap_seconds", "max"))
) if len(gaps) else pd.DataFrame(
    columns=["metric_id", "gaps", "affected_series", "median_gap_seconds", "max_gap_seconds"])
print(f"\ncollection gaps: {len(gaps):,}")
display(gap_summary)

if len(gaps):
    gap_scope = (gaps.groupby(
        ["entity_id", "episode_id", "gap_start", "gap_end"], as_index=False
    )["metric_id"].nunique().rename(columns={"metric_id": "metrics_affected"}))
    gap_scope_summary = (gap_scope.groupby("metrics_affected", as_index=False)
                         .size().rename(columns={"size": "gap_intervals"}))
else:
    gap_scope_summary = pd.DataFrame(columns=["metrics_affected", "gap_intervals"])
print("Gap scope: how many metrics disappear together")
display(gap_scope_summary)

## 4. Representative episodes

A reproducible hash sample of entities is used for plots. Within each entity
the episode closest to that entity's typical size and metric availability is
chosen, so the figures show typical behaviour rather than the largest file.
Population summaries stay in DuckDB. Pandas receives only a bounded, centred,
contiguous window from each sampled series, preserving temporal order without
putting the full telemetry table in memory.

In [ ]:
episode_sizes = series_summary.groupby(["entity_id", "episode_id"], as_index=False).agg(
    rows=("rows", "sum"), metrics=("metric_id", "nunique"),
    valid_values=("valid_values", "sum"),
)
episode_sizes["valid_rate"] = episode_sizes["valid_values"] / episode_sizes["rows"]

entities = sorted(episode_sizes["entity_id"].unique(),
                  key=lambda value: hashlib.sha256(str(value).encode()).hexdigest())
selected_entities = entities[:SAMPLE_ENTITIES]

typical = episode_sizes.loc[episode_sizes["entity_id"].isin(selected_entities)].copy()
reference = typical.groupby("entity_id")[["rows", "metrics"]].transform("median")
typical["distance"] = ((typical["rows"] - reference["rows"]).abs() / reference["rows"].clip(lower=1)
                       + (typical["metrics"] - reference["metrics"]).abs() / reference["metrics"].clip(lower=1))
selected_episodes = (typical.sort_values(["entity_id", "distance", "episode_id"])
                     .groupby("entity_id", as_index=False).head(1).reset_index(drop=True))
display(selected_episodes)

connection.register("selected_episodes_table", selected_episodes[["episode_id"]])
selected_series = series_summary.loc[
    series_summary["episode_id"].isin(selected_episodes["episode_id"])
]
points_per_series = max(1, min(
    MAX_TEST_POINTS, MAX_PANEL_ROWS // max(1, len(selected_series))
))
panel = sql(f"""
    WITH ranked AS (
        SELECT entity_id, episode_id, metric_id, event_ts, value, quality_code,
               row_number() OVER series_order AS position,
               count(*) OVER series_order AS series_rows
        FROM telemetry JOIN selected_episodes_table USING (episode_id)
        WINDOW series_order AS (
            PARTITION BY episode_id, metric_id ORDER BY event_ts
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )
    ), centred AS (
        SELECT *, floor(greatest(0, series_rows - {points_per_series}) / 2.0) + 1
                  AS first_position
        FROM ranked
    )
    SELECT entity_id, episode_id, metric_id, event_ts, value, quality_code
    FROM centred
    WHERE position >= first_position
      AND position < first_position + {points_per_series}
    ORDER BY entity_id, episode_id, metric_id, event_ts
""")
panel["event_ts"] = pd.to_datetime(panel["event_ts"], utc=True)
if len(panel) > MAX_PANEL_ROWS:
    raise MemoryError(f"Panel limit failed: {len(panel):,} > {MAX_PANEL_ROWS:,}")
print(f"Pandas panel: {len(panel):,} rows; at most {points_per_series:,} per series")
display(panel.head(10))
display(panel.pivot_table(
    index=["event_ts", "entity_id", "episode_id"],
    columns="metric_id", values="value", aggfunc="first",
).reset_index().head(6))


def series_of(episode_id, metric_id, valid_only=True):
    frame = panel.loc[panel["episode_id"].eq(episode_id) & panel["metric_id"].eq(metric_id)]
    if valid_only:
        numeric = pd.to_numeric(frame["value"], errors="coerce")
        frame = frame.loc[frame["quality_code"].ne("invalid") & np.isfinite(numeric)]
    return frame.set_index("event_ts")["value"].astype(float).sort_index()


cadence_rows = []
for (episode_id, metric_id), frame in panel.groupby(["episode_id", "metric_id"]):
    deltas = frame["event_ts"].diff().dt.total_seconds().dropna()
    if deltas.empty:
        continue
    expected = CADENCE.get(metric_id)
    cadence_limit = expected if pd.notna(expected) else deltas.median()
    cadence_rows.append({
        "episode_id": episode_id, "metric_id": metric_id, "observations": len(frame),
        "expected_cadence_seconds": expected,
        "median_delta_seconds": deltas.median(),
        "p95_delta_seconds": deltas.quantile(0.95),
        "irregular_rate": float((deltas > cadence_limit * GAP_TOLERANCE_FACTOR).mean()),
    })
cadence_summary = pd.DataFrame(cadence_rows)
display(cadence_summary.groupby("metric_id", as_index=False).agg(
    series=("episode_id", "nunique"),
    median_delta_seconds=("median_delta_seconds", "median"),
    irregular_rate=("irregular_rate", "mean"),
).round(4))

## 5. Distributions and representative series

Plots respect measurement kind: states use frequencies, interval counts use
a signed `log1p` transform, cumulative counters use within-segment increments, and
continuous measurements use observed levels. Nothing is interpolated, and a
line is broken wherever a collection gap was recorded.

In [ ]:
def transform(values, metric_id):
    """Return (series, label) in the representation that suits the kind."""

    kind = KIND.get(metric_id)
    if kind == "interval_count":
        return np.sign(values) * np.log1p(values.abs()), "signed_log1p"
    if kind == "cumulative_counter":
        return values.diff().dropna(), "increment"
    return values, "level"


def contiguous_segments(series, metric_id):
    """Split a series at collection gaps without dropping observations."""

    if len(series) < 2:
        return [series] if len(series) else []
    deltas = series.index.to_series().diff().dt.total_seconds()
    cadence = CADENCE.get(metric_id)
    limit = cadence if pd.notna(cadence) else deltas.median()
    block = deltas.gt(limit * GAP_TOLERANCE_FACTOR).cumsum()
    return [part for _, part in series.groupby(block)]


FIGURES.mkdir(parents=True, exist_ok=True)
ranked_metrics = metric_evidence.sort_values(
    ["availability", "constant_series_rate", "metric_id"],
    ascending=[False, True, True],
)
non_states = ranked_metrics.loc[ranked_metrics["measurement_kind"].ne("discrete_state")]
states = ranked_metrics.loc[ranked_metrics["measurement_kind"].eq("discrete_state")]
state_slots = min(MAX_STATE_PLOTS, len(states))
non_state_limit = MAX_PLOT_METRICS - state_slots
plot_metrics = non_states.drop_duplicates("measurement_kind")["metric_id"].tolist()
for metric_id in non_states["metric_id"]:
    if metric_id not in plot_metrics and len(plot_metrics) < non_state_limit:
        plot_metrics.append(metric_id)
plot_metrics = plot_metrics[:non_state_limit]
plot_metrics += states["metric_id"].head(state_slots).tolist()
plot_metrics = plot_metrics[:MAX_PLOT_METRICS]
print("Metrics selected for plots:", plot_metrics)

for metric_id in plot_metrics:
    kind = KIND[metric_id]
    figure, axes = plt.subplots(1, 2, figsize=(12, 2.8),
                                gridspec_kw={"width_ratios": [1, 2]})
    pooled_parts = []
    label = "level"
    for episode_id in selected_episodes["episode_id"]:
        for segment in contiguous_segments(series_of(episode_id, metric_id), metric_id):
            shown, label = transform(segment, metric_id)
            pooled_parts.append(shown)
    pooled = pd.concat(pooled_parts) if pooled_parts else pd.Series(dtype=float)
    if pooled.empty:
        plt.close(figure)
        continue

    if kind == "discrete_state":
        pooled.value_counts(normalize=True).sort_index().plot.bar(ax=axes[0], color="#3b6ea5")
        axes[0].set_ylabel("frequency")
    else:
        axes[0].hist(pooled.dropna(), bins=40, color="#3b6ea5")
        axes[0].set_ylabel(f"count ({label})")
    axes[0].set_title(f"{metric_id} — {kind}")

    for episode_id in selected_episodes["episode_id"].head(3):
        series = series_of(episode_id, metric_id)
        if series.empty:
            continue
        remaining = MAX_PLOT_POINTS
        for number, segment in enumerate(contiguous_segments(series, metric_id)):
            shown, label = transform(segment, metric_id)
            shown = shown.iloc[:remaining]
            if shown.empty:
                continue
            style = {"drawstyle": "steps-post"} if kind == "discrete_state" else {}
            axes[1].plot(shown.index, shown.values, linewidth=0.8,
                         label=episode_id[:28] if number == 0 else None, **style)
            remaining -= len(shown)
            if remaining <= 0:
                break
    axes[1].set_title(f"{metric_id} ({label})")
    axes[1].legend(fontsize=6, loc="upper right")
    figure.tight_layout()
    figure.savefig(FIGURES / f"metric_{metric_id}.png", dpi=110)
    plt.show()
    plt.close(figure)

## 6. Temporal dependence and seasonality

Diagnostics run on the **longest contiguous observed segment** of a series,
never on interpolated data. Counts use signed `log1p`, counters use increments, and
discrete states are excluded. STL runs only where the segment holds at least
six full cycles. ADF/KPSS lookup-table bounds are retained in `test_note`;
failed and insufficient tests remain explicit rows. These results are
evidence, not automatic modelling decisions.

In [ ]:
from statsmodels.tsa.stattools import acf, adfuller, kpss  # noqa: E402
from statsmodels.tsa.seasonal import STL  # noqa: E402


def longest_contiguous(series, metric_id):
    segments = contiguous_segments(series, metric_id)
    return max(segments, key=len) if segments else series


def seasonal_period(metric_id, length):
    """Daily cycle in samples, when the segment can actually support one."""

    cadence = CADENCE.get(metric_id)
    if not cadence or np.isnan(cadence):
        return None
    period = int(round(86400 / cadence))
    return period if 4 <= period <= length // 6 else None


temporal_rows, stationarity_rows, seasonality_rows = [], [], []
for metric_id in catalogue["metric_id"]:
    if KIND[metric_id] == "discrete_state":
        continue
    for episode_id in selected_episodes["episode_id"]:
        raw = series_of(episode_id, metric_id)
        if raw.empty:
            continue
        segment = longest_contiguous(raw, metric_id)
        values, label = transform(segment, metric_id)
        values = values.dropna()
        identity = {"episode_id": episode_id, "metric_id": metric_id}
        if len(values) < 30 or values.nunique() < 3:
            stationarity_rows.append({**identity, "status": "insufficient_data"})
            if KIND[metric_id] in CONTINUOUS:
                seasonality_rows.append({**identity, "status": "insufficient_data"})
            continue

        cadence = CADENCE.get(metric_id)
        daily_lag = (int(round(86400 / cadence))
                     if pd.notna(cadence) and cadence > 0 else None)
        target_lag = daily_lag if daily_lag and daily_lag <= 200 else 40
        max_lag = min(max(40, target_lag), 200, len(values) // 3)
        correlations = acf(values.to_numpy(), nlags=max_lag, fft=True)
        below = np.flatnonzero(np.abs(correlations[1:]) < 0.2) + 1
        temporal_rows.append({
            **identity, "representation": label, "segment_length": len(values),
            "max_lag_tested": max_lag, "acf_lag1": correlations[1],
            "acf_lag5": correlations[5] if max_lag >= 5 else np.nan,
            "daily_lag": daily_lag,
            "acf_daily": correlations[daily_lag]
            if daily_lag and daily_lag <= max_lag else np.nan,
            "first_below_0.2": int(below[0]) if len(below) else np.nan,
        })

        test_values = values.iloc[:MAX_TEST_POINTS].to_numpy()
        try:
            with warnings.catch_warnings(record=True) as test_warnings:
                warnings.simplefilter("always")
                max_adf_lag = min(40, max(1, len(test_values) // 10))
                adf_p = adfuller(test_values, maxlag=max_adf_lag, autolag="AIC")[1]
                kpss_p = kpss(test_values, regression="c", nlags="auto")[1]
        except (ValueError, np.linalg.LinAlgError) as error:
            stationarity_rows.append({
                **identity, "status": "test_failed", "error_type": type(error).__name__,
            })
        else:
            stationarity_rows.append({
                **identity, "status": "evaluated", "test_points": len(test_values),
                "test_note": " | ".join(str(item.message).replace("\n", " ")
                                           for item in test_warnings) or None,
                "adf_p": adf_p, "kpss_p": kpss_p,
                "verdict": ("stationary" if adf_p < 0.05 and kpss_p > 0.05
                            else "unit_root" if adf_p >= 0.05 and kpss_p <= 0.05
                            else "inconclusive"),
            })

        if KIND[metric_id] in CONTINUOUS:
            period = seasonal_period(metric_id, len(values))
            if period is None:
                seasonality_rows.append({
                    **identity, "status": "insufficient_cycles",
                    "segment_length": len(values),
                })
            else:
                try:
                    result = STL(values.to_numpy(), period=period, robust=True).fit()
                except (ValueError, np.linalg.LinAlgError) as error:
                    seasonality_rows.append({
                        **identity, "status": "fit_failed",
                        "error_type": type(error).__name__,
                    })
                else:
                    residual_variance = np.var(result.resid)
                    seasonal_variance = np.var(result.seasonal + result.resid)
                    trend_variance = np.var(result.trend + result.resid)
                    total_variance = np.var(values.to_numpy())
                    seasonality_rows.append({
                        **identity, "status": "evaluated", "period_samples": period,
                        "seasonal_strength": max(0.0, 1 - residual_variance / seasonal_variance)
                        if seasonal_variance else np.nan,
                        "trend_strength": max(0.0, 1 - residual_variance / trend_variance)
                        if trend_variance else np.nan,
                        "residual_share": residual_variance / total_variance
                        if total_variance else np.nan,
                    })

temporal_evidence = pd.DataFrame(temporal_rows)
stationarity_summary = pd.DataFrame(stationarity_rows)
seasonality_summary = pd.DataFrame(seasonality_rows)

if not temporal_evidence.empty:
    print("\nautocorrelation (median across sampled episodes)")
    display(temporal_evidence.groupby("metric_id")[[
        "acf_lag1", "acf_lag5", "acf_daily", "segment_length",
    ]].median().round(3))

if not stationarity_summary.empty:
    print("\nstationarity test status")
    display(pd.crosstab(stationarity_summary["metric_id"], stationarity_summary["status"]))
    evaluated_stationarity = stationarity_summary.loc[
        stationarity_summary["status"].eq("evaluated")
    ]
    if not evaluated_stationarity.empty:
        display(pd.crosstab(evaluated_stationarity["metric_id"],
                            evaluated_stationarity["verdict"]))

if not seasonality_summary.empty:
    print("\nseasonality test status")
    display(pd.crosstab(seasonality_summary["metric_id"], seasonality_summary["status"]))
    evaluated_seasonality = seasonality_summary.loc[
        seasonality_summary["status"].eq("evaluated")
    ]
    if not evaluated_seasonality.empty:
        display(evaluated_seasonality.groupby("metric_id")[[
            "seasonal_strength", "trend_strength", "residual_share",
        ]].median().round(3))

## 7. Cross-metric dependence

Only continuous metrics sharing one cadence are compared. Correlation is
computed **within** an episode and then summarised across episodes, so a
between-episode level shift never masquerades as a relationship. Spearman
correlation limits the influence of extreme values, and differences are reset
at every collection gap. Pair-specific counts show how much evidence supports
each estimate.

In [ ]:
comparable = catalogue.loc[
    catalogue["measurement_kind"].isin(CONTINUOUS)
].groupby("expected_cadence_seconds")["metric_id"].apply(list)

dependence_rows = []
for cadence, metrics in comparable.items():
    if len(metrics) < 2:
        continue
    for episode_id in selected_episodes["episode_id"]:
        wide = pd.DataFrame({m: series_of(episode_id, m) for m in metrics}).dropna(how="all")
        if len(wide) < 30:
            continue
        wide = wide.sort_index()
        changes = wide.diff()
        breaks = wide.index.to_series().diff().gt(
            pd.Timedelta(seconds=float(cadence) * GAP_TOLERANCE_FACTOR)
        )
        changes.loc[breaks.to_numpy(), :] = np.nan
        levels = wide.corr(method="spearman", min_periods=30)
        change_correlations = changes.corr(method="spearman", min_periods=30)
        for i, first in enumerate(metrics):
            for second in metrics[i + 1:]:
                if first not in levels or second not in levels:
                    continue
                level_n = int(wide[[first, second]].dropna().shape[0])
                difference_n = int(changes[[first, second]].dropna().shape[0])
                if level_n < 30:
                    continue
                dependence_rows.append({
                    "episode_id": episode_id, "metric_a": first, "metric_b": second,
                    "level_observations": level_n,
                    "difference_observations": difference_n,
                    "level_corr": levels.loc[first, second],
                    "difference_corr": change_correlations.loc[first, second]
                    if difference_n >= 30 else np.nan,
                })

dependence_evidence = pd.DataFrame(dependence_rows)
if not dependence_evidence.empty:
    pairs = dependence_evidence.groupby(["metric_a", "metric_b"], as_index=False).agg(
        episodes=("episode_id", "nunique"),
        level_observations=("level_observations", "sum"),
        difference_observations=("difference_observations", "sum"),
        level_corr_median=("level_corr", "median"),
        difference_corr_median=("difference_corr", "median"))
    display(pairs.reindex(pairs["level_corr_median"].abs().sort_values(ascending=False).index)
            .head(20).round(3))
else:
    print("No comparable same-cadence continuous pair had enough overlap.")

## 8. Save compact evidence

Summary tables and figures, not another copy of telemetry.

In [ ]:
artifacts = {
    "structure": structure,
    "selected_episodes": selected_episodes,
    "metric_evidence": metric_evidence,
    "series_summary": series_summary,
    "cadence_summary": cadence_summary,
    "gap_summary": gap_summary,
    "gap_scope_summary": gap_scope_summary,
    "temporal_evidence": temporal_evidence,
    "stationarity_summary": stationarity_summary,
    "seasonality_summary": seasonality_summary,
    "dependence_evidence": dependence_evidence,
}
summary = {
    "eda_version": EDA_VERSION,
    "sector": SECTOR,
    "core_fingerprint": core_manifest["fingerprint"],
    "core_row_counts": core_manifest["row_counts"],
    "sampled_entities": selected_entities,
    "plotted_metrics": plot_metrics,
    "pandas_panel_rows": len(panel),
    "points_per_sampled_series": points_per_series,
    "maximum_pandas_rows": MAX_PANEL_ROWS,
    "rows": {name: len(frame) for name, frame in artifacts.items()},
    "metrics_by_kind": catalogue["measurement_kind"].value_counts().to_dict(),
    "collection_gaps": len(gaps),
}

with new_output_directory(EDA_ROOT) as output:
    for name, frame in artifacts.items():
        frame.to_parquet(output / f"{name}.parquet", index=False)
    shutil.copytree(FIGURES, output / "figures")
    write_json(output / "eda_summary.json", summary)

print("EDA evidence:", EDA_ROOT)
for name, frame in artifacts.items():
    print(f"  {name:24s} {len(frame):>7,} rows")
print("Next: 03_EVALUATION_HARNESS.ipynb")
connection.close()
duckdb_spill.cleanup()
figure_cache.cleanup()